In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10

In [2]:
# Datasets and DataLoader
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image -> scale -> normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = CIFAR10(root="./data", train= True, download=True, transform=transform)
testset = CIFAR10(root="./data", train= False, download=True, transform=transform)

In [3]:
train_loader = DataLoader(trainset, batch_size=64, shuffle=True)
test_loader = DataLoader(testset, batch_size=64)

### Building the CNN

In [5]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            # first convolution layer
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # Kernel size = 2 & stride = 2

            # second convolution layer
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # third convolution layer
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # Flattening
        x = self.fc_layers(x)

        return x

In [6]:
model = CNN()

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training the CNN

In [11]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()

        output = model.forward(images) # FP
        loss = criterion(output, labels) # Loss function
        loss.backward() # BP
        optimizer.step() # Update parameters

        epoch_training_loss += loss.item()

    print(f"Epoch = {epoch + 1}/{epochs} & Loss = {epoch_training_loss/len(train_loader)}")

Epoch = 1/10 & Loss = 0.9483163858313695
Epoch = 2/10 & Loss = 0.7595771835054583
Epoch = 3/10 & Loss = 0.6339122192252933
Epoch = 4/10 & Loss = 0.5286105022291698
Epoch = 5/10 & Loss = 0.4354642782636616
Epoch = 6/10 & Loss = 0.3504788983718056
Epoch = 7/10 & Loss = 0.27411256335161227
Epoch = 8/10 & Loss = 0.21245194450401894
Epoch = 9/10 & Loss = 0.16910404812001512
Epoch = 10/10 & Loss = 0.13740391790619133


In [12]:
# Evaluation
correct_preds = 0
total_labels = 0

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_preds += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy = {(correct_preds/total_labels) * 100} %")

Accuracy = 74.9 %
